# W4 — Data Processing with pandas

## Goal
Transform raw environmental observations into an analysis-ready table
while keeping every processing decision traceable.

Workflow:

Raw → Inspect → Parse → Transform → Aggregate → Audit → Processed

In [1]:
# load data
import pandas as pd
from pathlib import Path

DATA_FILE = Path("../data/sample/zhongli_aq_2025.csv")

df = pd.read_csv(DATA_FILE)

print("Shape:", df.shape)
display(df.head())

Shape: (168, 8)


,datetime,station,PM2.5,O3,NO2,TEMP,RH,WS
0,2025-01-01 00:00:00,Zhongli,14.3,24.8,20.8,13.0,78.5,1.6
1,2025-01-01 01:00:00,Zhongli,13.3,25.9,21.6,14.0,75.3,3.6
2,2025-01-01 02:00:00,Zhongli,9.1,15.4,27.2,13.0,97.9,1.2
3,2025-01-01 03:00:00,Zhongli,14.0,18.9,29.8,13.2,81.4,1.0
4,2025-01-01 04:00:00,Zhongli,13.9,6.6,27.8,13.4,79.0,1.4


In [2]:
# p.35, select -> filter -> assign
cols = ["datetime", "station", "PM2.5", "O3", "WS"]
x = df[cols].copy()

display(x.head())
print("Shape:", x.shape)

,datetime,station,PM2.5,O3,WS
0,2025-01-01 00:00:00,Zhongli,14.3,24.8,1.6
1,2025-01-01 01:00:00,Zhongli,13.3,25.9,3.6
2,2025-01-01 02:00:00,Zhongli,9.1,15.4,1.2
3,2025-01-01 03:00:00,Zhongli,14.0,18.9,1.0
4,2025-01-01 04:00:00,Zhongli,13.9,6.6,1.4


Shape: (168, 5)


In [3]:
# p.35, parse datetime
print(x["datetime"].dtype)

object


In [3]:
x["datetime"] = pd.to_datetime(x["datetime"])

print(x["datetime"].dtype)

datetime64[ns]


In [4]:
# p.35 -> filter
x = x.loc[x["station"].eq("Zhongli")]
print("Before filtering:", len(x))

x = x.loc[x["station"].eq("Zhongli")].copy()

print("After filtering:", len(x))

Before filtering: 168
After filtering: 168


In [5]:
# p.35, assign
x["PM2.5"].describe()

count     167.000000
mean      -34.743114
std       775.846560
min     -9999.000000
25%        19.050000
50%        23.800000
75%        30.950000
max       185.000000
Name: PM2.5, dtype: float64

In [6]:
x.loc[x["PM2.5"] < 0, ["datetime", "PM2.5"]]

,datetime,PM2.5
18,2025-01-01 18:00:00,-9999.0
57,2025-01-03 09:00:00,-2.0


In [7]:
x["PM2.5_valid"] = x["PM2.5"].where(x["PM2.5"] >= 0)

display(
    x.loc[x["PM2.5"] < 0,
          ["datetime", "PM2.5", "PM2.5_valid"]]
)

,datetime,PM2.5,PM2.5_valid
18,2025-01-01 18:00:00,-9999.0,NaN
57,2025-01-03 09:00:00,-2.0,NaN


In [8]:
# p.36 groupby vs. resample
station_mean = (
    x.groupby("station")["PM2.5_valid"]
     .mean()
)

display(station_mean)

station
Zhongli    25.447879
Name: PM2.5_valid, dtype: float64

In [9]:
xt = x.set_index("datetime")

display(xt.head())

,station,PM2.5,O3,WS,PM2.5_valid
datetime,,,,,
2025-01-01 00:00:00,Zhongli,14.3,24.8,1.6,14.3
2025-01-01 01:00:00,Zhongli,13.3,25.9,3.6,13.3
2025-01-01 02:00:00,Zhongli,9.1,15.4,1.2,9.1
2025-01-01 03:00:00,Zhongli,14.0,18.9,1.0,14.0
2025-01-01 04:00:00,Zhongli,13.9,6.6,1.4,13.9


In [10]:
daily_mean = xt["PM2.5_valid"].resample("D").mean()
#daily_mean = xt["PM2.5_valid"].resample("ME").mean()

display(daily_mean)

datetime
2025-01-01    26.034783
2025-01-02    23.183333
2025-01-03    23.821739
2025-01-04    24.347826
2025-01-05    24.150000
2025-01-06    30.754167
2025-01-07    25.754167
Freq: D, Name: PM2.5_valid, dtype: float64

In [11]:
# p.37, count
daily = xt["PM2.5_valid"].resample("D").agg(
    valid_hours="count",
    mean_pm25="mean"
)

display(daily)

,valid_hours,mean_pm25
datetime,,
2025-01-01,23,26.034783
2025-01-02,24,23.183333
2025-01-03,23,23.821739
2025-01-04,23,24.347826
2025-01-05,24,24.150000
2025-01-06,24,30.754167
2025-01-07,24,25.754167


In [12]:
#MIN_VALID_HOURS = 18
MIN_VALID_HOURS = 24

daily["complete"] = daily["valid_hours"] >= MIN_VALID_HOURS

display(daily)

,valid_hours,mean_pm25,complete
datetime,,,
2025-01-01,23,26.034783,False
2025-01-02,24,23.183333,True
2025-01-03,23,23.821739,False
2025-01-04,23,24.347826,False
2025-01-05,24,24.150000,True
2025-01-06,24,30.754167,True
2025-01-07,24,25.754167,True


In [21]:
daily["PM2.5_daily"] = daily["mean_pm25"].where(
    daily["complete"]
)

display(daily)

,valid_hours,mean_pm25,complete,PM2.5_daily
datetime,,,,
2025-01-01,23,26.034783,False,NaN
2025-01-02,24,23.183333,True,23.183333
2025-01-03,23,23.821739,False,NaN
2025-01-04,23,24.347826,False,NaN
2025-01-05,24,24.150000,True,24.150000
2025-01-06,24,30.754167,True,30.754167
2025-01-07,24,25.754167,True,25.754167


In [13]:
# p.39, insert two tables
left = pd.DataFrame({
    "station": ["Zhongli", "Zhongli"],
    "datetime": pd.to_datetime([
        "2026-10-01 10:00",
        "2026-10-01 11:00"
    ]),
    "PM2.5": [25, 30]
})

right = pd.DataFrame({
    "station": ["Zhongli", "Zhongli", "Zhongli"],
    "datetime": pd.to_datetime([
        "2026-10-01 10:00",
        "2026-10-01 10:00",
        "2026-10-01 11:00"
    ]),
    "WS": [1.5, 1.7, 2.0]
})

In [14]:
display(left)
display(right)

,station,datetime,PM2.5
0,Zhongli,2026-10-01 10:00:00,25
1,Zhongli,2026-10-01 11:00:00,30


,station,datetime,WS
0,Zhongli,2026-10-01 10:00:00,1.5
1,Zhongli,2026-10-01 10:00:00,1.7
2,Zhongli,2026-10-01 11:00:00,2.0


In [15]:
bad_merge = left.merge(
    right,
    on=["station", "datetime"]
)

display(bad_merge)

print("Left rows:", len(left))
print("Merged rows:", len(bad_merge))

,station,datetime,PM2.5,WS
0,Zhongli,2026-10-01 10:00:00,25,1.5
1,Zhongli,2026-10-01 10:00:00,25,1.7
2,Zhongli,2026-10-01 11:00:00,30,2.0


Left rows: 2
Merged rows: 3


In [16]:
# p.39, validate
left.merge(
    right,
    on=["station", "datetime"],
    validate="one_to_one"
)

MergeError: Merge keys are not unique in right dataset; not a one-to-one merge

In [17]:
right.duplicated(
    ["station", "datetime"]
).sum()

np.int64(1)

In [18]:
right.loc[
    right.duplicated(
        ["station", "datetime"],
        keep=False
    )
]

,station,datetime,WS
0,Zhongli,2026-10-01 10:00:00,1.5
1,Zhongli,2026-10-01 10:00:00,1.7


In [19]:
# p.40, autit table
audit = pd.DataFrame({
    "Step": [
        "Raw load",
        "Parse time",
        "Mask negative PM2.5",
        "Daily aggregation"
    ],
    "Rows": [
        len(df),
        len(x),
        len(x),
        len(daily)
    ],
    "Notes": [
        "Original teaching sample",
        f"{x['datetime'].isna().sum()} unparsed timestamps",
        f"{x['PM2.5_valid'].isna().sum()} invalid/missing values",
        f"Require >= {MIN_VALID_HOURS} valid hours"
    ]
})

display(audit)

,Step,Rows,Notes
0,Raw load,168,Original teaching sample
1,Parse time,168,0 unparsed timestamps
2,Mask negative PM2.5,168,3 invalid/missing values
3,Daily aggregation,7,Require >= 24 valid hours


In [20]:
# p.41, raw ≠ processed, hourly

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(exist_ok=True)

x.to_csv(
    PROCESSED_DIR / "zhongli_hourly_processed.csv",
    index=False
)

In [21]:
# daily
daily.to_csv(
    PROCESSED_DIR / "zhongli_daily_pm25.csv"
)

In [22]:
# aduit
audit.to_csv(
    PROCESSED_DIR / "processing_audit.csv",
    index=False
)